# Shreve Week 04 — Stochastic Calculus — Integrands

**Week 4** — Stochastic Calculus — Integrands

This notebook teaches **stochastic calculus — integrands** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **Simple processes & Itô integral** | Ch. 4.1–4.2 |
| 2 | **Itô isometry** | Ch. 4.2 |
| 3 | **Quadratic variation of $W$** | Ch. 4.2 |
| 4 | **$L^2$ integrands** | Ch. 4.3 |
| — | **Problem set** | Ch. 4 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 4 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Simple Integrands and the Itô Integral

A **simple process** $H_t = \sum_i H_i \mathbf{1}_{(t_i, t_{i+1}]}(t)$ is piecewise constant on intervals.

**Itô integral:**

$$\int_0^T H_t\, dW_t = \sum_i H_i (W_{t_{i+1}} - W_{t_i})$$

Evaluated at **left endpoints** — this is the key difference from Stratonovich.

**Shreve Ch. 4.1–4.2:** construction for simple integrands.


In [ ]:
def ito_integral_simple(H_values, T=1.0, seed=30):
    rng = np.random.default_rng(seed)
    n = len(H_values)
    dt = T / n
    dW = rng.normal(0, np.sqrt(dt), size=n)
    integral = np.sum(H_values * dW)
    return integral

H = np.ones(1000)
ints = [ito_integral_simple(H, seed=s) for s in range(500)]
print(f"∫₀¹ 1 dW_t: mean={np.mean(ints):.4f}, var={np.var(ints):.4f} (theory 0, 1)")


---
# Part 2 — Itô Isometry

For simple $H$:

$$E\left[\left(\int_0^T H_t\, dW_t\right)^2\right] = E\left[\int_0^T H_t^2\, dt\right]$$

**Shreve Ch. 4.2:** Itô isometry extends integral to $L^2$ integrands.


In [ ]:
# Itô isometry: Var(∫ H dW) = E[∫ H² dt]
rng = np.random.default_rng(31)
T, n = 1.0, 500
dt = T / n
n_sims = 10_000
H = rng.uniform(0, 1, size=n)  # random simple integrand

integrals = np.array([
    np.sum(H * rng.normal(0, np.sqrt(dt), size=n))
    for _ in range(n_sims)
])
var_sim = integrals.var()
theory = np.sum(H**2) * dt
print(f"Var(∫H dW) sim = {var_sim:.4f}, E[∫H²dt] = {theory:.4f}")


---
# Part 3 — Quadratic Variation of Integrals

For $I_t = \int_0^t H_s\, dW_s$, $[I, I]_t = \int_0^t H_s^2\, ds$.

Combined with $W$: $W_t^2 - t$ is a martingale (preview of Itô's lemma).

**Shreve Ch. 4.2:** quadratic variation of Itô integrals.


In [ ]:
# W_t² - t has zero mean (martingale)
rng = np.random.default_rng(32)
T, n_steps, n_sims = 1.0, 500, 20_000
dt = T / n_steps
means = []
for _ in range(n_sims):
    dW = rng.normal(0, np.sqrt(dt), size=n_steps)
    W = np.cumsum(dW)
    W_T = W[-1]
    means.append(W_T**2 - T)
print(f"E[W_T² - T] = {np.mean(means):.4f} (theory 0)")


---
# Part 4 — Extending to $L^2$ Integrands

Process $H$ is in $\mathcal{L}^2$ if $E[\int_0^T H_t^2 dt] < \infty$. Approximate by simple processes; integral defined via Itô isometry limit.

**Shreve Ch. 4.3:** closure of simple integrands in $L^2$.


In [ ]:
# ∫₀¹ W_t dW_t via discrete approximation (left endpoints)
def ito_integral_W(seed=33):
    rng = np.random.default_rng(seed)
    T, n = 1.0, 2000
    dt = T / n
    dW = rng.normal(0, np.sqrt(dt), size=n)
    W = np.concatenate([[0], np.cumsum(dW)])
    # H_t = W_{t_i} on (t_i, t_{i+1}]
    integral = np.sum(W[:-1] * dW)
    return integral

ints = [ito_integral_W(seed=s) for s in range(5000)]
# Itô: ∫ W dW = (W_T² - T)/2
print(f"∫ W dW: mean={np.mean(ints):.4f}, var={np.var(ints):.4f}")
print("Itô lemma predicts E[∫ W dW] = 0")


**Your turn:** Compare left-endpoint vs midpoint (Stratonovich) for $\int W\, dW$. Which gives zero mean?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For constant $H_t = c$, show $E[\int_0^T c\, dW_t] = 0$ and $\text{Var} = c^2 T$.

2. Prove Itô isometry for a simple process with non-random $H_i$.

3. Show $M_t = W_t^2 - t$ is a martingale (compute $E[W_{t+\Delta}^2 \mid W_t]$).

4. Why must we use left endpoints in the Itô integral definition?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Integral is $c W_T$, mean 0, var $c^2 T$.

**2.** Expand square of sum; cross terms have zero expectation by independent increments.

**3.** $E[W_{t+\Delta}^2] = W_t^2 + \Delta$, so $E[W_{t+\Delta}^2 - (t+\Delta) \mid W_t] = W_t^2 - t$.

**4.** Left endpoints make increment $W_{t_{i+1}}-W_{t_i}$ independent of $H_{t_i}$; midpoint would correlate $H$ with future increment.

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 4 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
